In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Sinal de notícias por evento · preço + texto (Colab)

Ative GPU no Colab antes de executar. Este notebook clona ou atualiza o repositório no GitHub, monta o Drive e lê `MyDrive/latent_fusion_data/fusion_event_panel.npz` e `fusion_events.jsonl.gz`. Salva os embeddings MiniLM congelados em cache no Drive, ajusta o baseline de preços no treino (validação para parada antecipada), congela seus parâmetros e ajusta **somente** o residual de notícias. Nada é treinado no computador local.

Manchetes são avaliadas apenas depois do pregão extra de atraso presente em `decision_date`. O feed bruto tem corpos de artigos ocasionalmente trocados entre manchetes/URLs; o manifesto contém somente manchetes verificadas, sem corpos. A ausência de notícia não prova inexistência de notícias. Há lacuna completa em janeiro de 2024 e cobertura posterior incompleta: o teste auditável termina em **30/11/2023**, sem inferências sobre 2024+. O universo ex post de 49 ativos, incluindo ETFs, pode ter viés de seleção/sobrevivência. Retornos de cinco pregões consecutivos se sobrepõem; métricas/ablação não comprovam causalidade.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute exclusivamente no Google Colab com GPU') from exc
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==2.3.1', 'transformers>=4.33.3,<5'], check=True)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('O runtime Colab não disponibiliza PyTorch; selecione GPU e reinicie')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative GPU: Ambiente de execução → Alterar tipo de ambiente → T4 GPU')
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
PANEL, NEWS = DATA / 'fusion_event_panel.npz', DATA / 'fusion_events.jsonl.gz'
for file in (PANEL, NEWS):
    if not file.is_file():
        raise FileNotFoundError(f'Envie o artefato gerado para {file}')
OUT = DATA / 'event_signal_results'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
{'gpu': torch.cuda.get_device_name(0), 'painel': str(PANEL), 'notícias': str(NEWS), 'saída': str(OUT)}

## Treino e avaliação fora da amostra

O alvo é o retorno simples do ativo em cinco pregões menos o retorno igualmente ponderado do universo no mesmo horizonte. O modelo residual tem valor **exatamente igual** ao baseline quando não há eventos recentes; isso não resolve falta de cobertura do feed. A ablação troca ticker e atrasa eventos, sem utilizar futuro. Os pesos da estratégia são positivos e somam 1; o rebalanceamento é agendado a cada cinco pregões **após** o fechamento, com 10 bps por valor negociado (compras + vendas). Se faltar previsão para algum ativo na data agendada, a carteira mantém os pesos até o próximo rebalanceamento; o relatório contabiliza essas decisões não executadas. O BH compra pesos iguais na mesma primeira data e conserva as quantidades, também pagando custo inicial.


In [ ]:
from IPython.display import display, Image
from src.fusion.event_experiment import run_experiment

result = run_experiment(PANEL, NEWS, OUT, cache_dir=DATA / 'article_embedding_cache', cost_bps=10)
display(result['signal'], result['portfolio'], result['price_history'], result['event_history'])
display(result['coverage'].pivot(index='month', columns='ticker', values='events'))
display(Image(filename=str(OUT / 'event_vs_bh.png')))
{'relatório': str(OUT / 'results.md'), 'curvas': str(OUT / 'equity_curves.csv'), 'cache': str(result['cache'])}